In [ ]:
%cd d:\Troida

import sys, os
project_root = os.path.abspath(".")
if project_root not in sys.path: sys.path.insert(0, project_root)


In [ ]:
from types import SimpleNamespace

# настройки MACD/catboost, меняйте при необходимости
SETTINGS = SimpleNamespace(
    csv="./data/21.10.2024 13.00.00 - 21.10.2025 13.00.00/YDEX_2H_20241021_130000_20251021_130000.csv",
    lookback=1000,
    horizon=100,
    step=20,
    cost_bps=5.0,
    macd_fast="",
    macd_slow="",
    macd_signal="",
    cat_overrides="",
)


In [ ]:
from types import SimpleNamespace
from macd import run_pipeline, compute_macd
import matplotlib.pyplot as plt

args = SimpleNamespace(
    csv=SETTINGS.csv,
    lookback=SETTINGS.lookback,
    horizon=SETTINGS.horizon,
    step=SETTINGS.step,
    cost_bps=SETTINGS.cost_bps,
    macd_fast=SETTINGS.macd_fast,
    macd_slow=SETTINGS.macd_slow,
    macd_signal=SETTINGS.macd_signal,
    cat_overrides=SETTINGS.cat_overrides,
    plot=True,
)
result = run_pipeline(args, verbose=True)

df = result["df"]
best_params = result["best_params"]
macd_full = compute_macd(df["close"], best_params["fast"], best_params["slow"], best_params["signal"])

fig, ax = plt.subplots(3, 1, sharex=True, figsize=(12, 8))
df["close"].plot(ax=ax[0], title="Ценовой график")
ax[0].grid(True)

macd_full[["macd", "signal"]].plot(ax=ax[1], title=f"MACD ({best_params['fast']},{best_params['slow']},{best_params['signal']})")
ax[1].grid(True)

macd_full["hist"].plot(ax=ax[2], color="tab:purple", title="MACD Histogram")
ax[2].grid(True)

plt.tight_layout()
plt.show()
print("Лучшие параметры MACD:", best_params)
